In [ ]:
!pip install -U bitsandbytes # quantization

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 10.6 MB/s eta 0:00:00


In [ ]:
!pip install trl #transformer reinforment learning(SFT:superwised finetuning)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 21.0 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
import torch
torch.cuda.is_available()

True

In [ ]:
import os
import torch
from datasets import load_dataset



In [ ]:
#AutoModelForCausalLM : for encoder only model we cant get a context vector for that perpose we use head or we can say for text generation it will used.
#autotokenizer : for downloading tokenizer
# BitsAndBytesConfig : quantization perpose
#TrainingArguments : for training we provide args here.
#pipeline: infernce use it will provide all things.

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

Dataset

In [ ]:
# the instruction dataset to use
dataset_name = "mlabonne/guanaco-llama2-1k"

# load dataset
dataset = load_dataset(dataset_name, split="train")

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Examples of creating datasets:

- system message : act as a AI engineer
- Human message : explain lora to me
- ai message : low rank adaption is ...
------------------------------------------------
- system message : act as a ML engineer
- Human message : explain linear regression to me
- ai message : linear regression is ...

In [ ]:
dataset.column_names

['text']

In [ ]:
dataset['text'][0]

'<s>[INST] Me gradué hace poco de la carrera de medicina ¿Me podrías aconsejar para conseguir rápidamente un puesto de trabajo? [/INST] Esto vale tanto para médicos como para cualquier otra profesión tras finalizar los estudios aniversarios y mi consejo sería preguntar a cuántas personas haya conocido mejor. En este caso, mi primera opción sería hablar con otros profesionales médicos, echar currículos en hospitales y cualquier centro de salud. En paralelo, trabajaría por mejorar mi marca personal como médico mediante un blog o formas digitales de comunicación como los vídeos. Y, para mejorar las posibilidades de encontrar trabajo, también participaría en congresos y encuentros para conseguir más contactos. Y, además de todo lo anterior, seguiría estudiando para presentarme a las oposiciones y ejercer la medicina en el sector público de mi país. </s>'

## BitsandBytesConfig for quantization

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
)
'''
load_in_4bit : we are loading the base model with a 4 bit quantization,
              so we are seting this value to true
'''

'\nload_in_4bit : we are loading the base model with a 4 bit quantization, \n              so we are seting this value to true\n'

## Model and Tokenizer

In [ ]:
# the model name you want to train from hugging face hub

model_name = "meta-llama/Llama-2-7b-chat-hf"

In [ ]:
from huggingface_hub import notebook_login

# Replace 'hf_YOUR_TOKEN_HERE' with your actual Hugging Face token
# You can also store your token securely as a Colab secret and access it using userdata.get('HF_TOKEN')
notebook_login()

In [ ]:
# load LLama Tokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-2-7b-chat-hf")

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [ ]:
#load base model

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
)

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/188 [00:00<?, ?B/s]

Lora Config

In [ ]:
## Working of lora:
1) create a copy of weight matrix(dw)
2) freeze the orignal weight matrix (W)
3) decompose dw into low dimensions: (A & B)

## Finetune
finetune (update) decomposed weight matrix (A & B)

In [ ]:
# Load Lora Configuration
peft_config = LoraConfig(
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

In [ ]:
# # 500 * 500
# 500 * 64, 64*500

In [ ]:
'''
The LoraConfig has the following attributes.

>> lora_dropout: dropout probability of the LoRA layers.
          This parameter is used to avoid overfitting.
           This technique basically drop-outs some of the neurons during both
           forward and backward propagation, this will help in removing dependency
           on a single unit of neurons. We are setting this to 0.1 (which is 10%),
           which means each neuron has a dropout chance of 10%.
>> r: This is the dimension of the low-rank matrix,
      In this case, we are setting this to 64
       (which effectively means we will have 512x64 and 64x512 parameters in our LoRA adapter.
       this is hyperparameter, its not fix, to use 64 or something else.
>> bias: We will not be training the bias in this example, so we are setting that to “none”.
        If we have to train the biases, we can set this to “all”, or if we want to train
        only the LORA biases then we can use “lora_only”
        cause when we changes the bias values then it will changes so much in our outputs, thats why we make them none
>> task_type: Since we are using the Causal language model, the task type we set to CAUSAL_LM.
'''

In [ ]:
from peft import  get_peft_model

### it will copy our orignal weights and decompose to A and B

In [ ]:
lora_model = get_peft_model(model, peft_config)

Training Arguments

In [ ]:
# Set training parameters
training_arguments = TrainingArguments(
    output_dir = "/content/LoRA",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=0,
    learning_rate=2e-4,
    weight_decay=0.001,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_steps=0.03,
    report_to="none"
)

In [ ]:
'''
>> output_dir: The output directory is where the model predictions and checkpoints will be stored.
>> num_train_epochs: One training epoch.
>> per_device_train_batch_size: Batch size per GPU for training.
>> gradient_accumulation_steps: This refers to the number of steps required to accumulate the gradients during the update process.
>> Optim : Model optimizer (AdamW optimizer).
>> save_steps: How often to save a checkpoint of the model during training.
>> learning_rate: Initial learning rate.
>> weight_decay: Regularization technique applied to model weights to prevent overfitting.
>> max_grad_norm: Gradient clipping.
>> max_steps: Number of training steps.
>> warmup_ratio: Ratio of steps for a linear warmup.
>> group_by_length: This can significantly improve performance and accelerate the training process.
                    When set to `True`, samples of similar lengths are grouped together in
                    batches, making training more efficient. This avoids wasting computation
                    on padding tokens when sequences of different lengths are processed together.
'''

### Supervised Fine-Tining(SFT)

In [ ]:
'''
Supervised Fine-Tuning(SFT) :
The TRL library from Hugging FAce provides an easy-to-use API to create SFT models
and train them on datasets.
'''

In [ ]:
# set superwised fine-tuning parameters

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_config,
    args=training_arguments,
)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
500,1.324014


In [ ]:
# Save Finetunned Model :
trainer.save_model("/content/LoRA")

## Merging Base Model and finetuned weights:

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
   model_name,
   quantization_config = bnb_config,
)

model = PeftModel.from_pretrained(base_model, "/content/LoRA")

# Reload tokeniser to save it

tokenizer = AutoTokenizer.from_pretrained(model_name)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
## Adapter :

>> Financial use case :
finance_model = PeftModel.from_pretrained(base_model, "/content/Finace_model")

>> HR               :
finance_model = PeftModel.from_pretrained(base_model, "/content/Hr_model")


In [ ]:
## Benefits of Using LoRA :
1) Original knowledge is preserved :
2) Less resources required
3) we can use same base model for different tasks (adapter)

## Difference Between QLora and Lora

In [ ]:
# LoRA :
it apply low-rank approximation to the weight of update matrix (delta w)

reduces memory requirements : it significataly reduces parameters
fast fine-tuning speed : compared to tradational finetuning it takes less memory and train faster
performance : LoRA maintain almost similar performance compared to tradational fine tuned model

In [ ]:
# QLoRA :
It take takes memory reduction to next level by quatizing adapter(A & B).

more momory reduction :
performance : almost similar to lora (slightly degrade)

In [ ]:
from huggingface_hub import notebook_login

# Replace 'hf_YOUR_TOKEN_HERE' with your actual Hugging Face token
# You can also store your token securely as a Colab secret and access it using userdata.get('HF_TOKEN')
notebook_login()

In [ ]:
model.push_to_hub("sangamw/finetunedLLamaModel7b")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   1%|          | 1.23MB /  134MB            

CommitInfo(commit_url='https://huggingface.co/sangamw/finetunedLLamaModel7b/commit/05e8b4ffe3e2f9ac3649071a74a42e15c0723a7c', commit_message='Upload model', commit_description='', oid='05e8b4ffe3e2f9ac3649071a74a42e15c0723a7c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/sangamw/finetunedLLamaModel7b', endpoint='https://huggingface.co', repo_type='model', repo_id='sangamw/finetunedLLamaModel7b'), pr_revision=None, pr_num=None)